<a href="https://colab.research.google.com/github/KevinCY-Kim/AI_Study/blob/main/ChuGyouk_PubMedQA_LLM_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ChuGyouk_PubMedQA 데이터셋을 활용한 LLM Finetuning
 - 4비트 사전 양자화된 모델"unsloth/Meta-Llama-3.1-8B" 활용

### LoRA 설정 (경량·속도 최적화형)
 - 목표: 작은 데이터셋 기준, VRAM 절약 + 빠른 학습 속도
 - r=8: LoRA 랭크 낮게 설정 → 적은 파라미터 학습, 과적합 방지
 - target_modules=["q_proj", "v_proj"]: 핵심 어텐션 부분만 학습 → 속도·메모리 절감
 - lora_alpha=16: LoRA 영향력 중간 수준 (기본 대비 안정적 수렴)
 - lora_dropout=0.05: 약간의 규제 효과로 일반화 향상
 - bias="none": 불필요한 bias 학습 비활성화
 - use_gradient_checkpointing="unsloth": GPU 메모리 절약, 긴 시퀀스 학습 가능
 - use_rslora=False: 안정성 모드는 해제 (속도 우선)
##### “LoRA r=8, q/v만 학습 — 메모리 절약형 설정 (Unsloth 체크포인트 활성화, 빠른 미세조정용)

### 프롬프트 & 종료 토큰
 - 목적: 모델이 학습 시 “질문 → 답변” 구조를 명확히 이해하고,
문장 끝에서 생성이 멈추도록 EOS 토큰을 추가함.

In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "ChuGyouk/PubMedQA-test-Ko",
    split="test",
    cache_dir="/content"  # 예: 구글 코랩이나 로컬 폴더
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/155 [00:00<?, ?B/s]

pubmedQA_test_translated.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

In [1]:
!pip install unsloth

In [ ]:
!pip list

Package                                  Version
---------------------------------------- --------------------
absl-py                                  1.4.0
absolufy-imports                         0.3.1
accelerate                               1.10.1
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.0
aiosignal                                1.4.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.17.0
altair                                   5.5.0
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                                    4.11.0
anywidget                                0.9.18
argon2-cffi                              25.1.0
argon2-cffi-bindings              

In [3]:
# Unsloth 라이브러리에서 FastLanguageModel을 임포트
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # 최대 시퀀스 길이를 설정 ( 텍스트의 최대 길이를 지정)
dtype = None  # 자동 감지를 위해 None 설정. Tesla T4는 Float16, Ampere+는 Bfloat16 사용. 모델의 파라미터를 저장할 데이터 타입
load_in_4bit = True  # 메모리 사용량을 줄이기 위해 4비트 양자화 사용. 다만 양자화에 따른 손실이 있어서 필요에 따라 False로 설정 가능

# 4비트 사전 양자화된 모델 리스트 (더 빠른 다운로드 및 OOM 방지)
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 8B 모델, 4비트 양자화
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 405B 모델도 4비트 지원
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # Mistral 12B 모델, 4비트 지원
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 7B 모델, 4비트 지원
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 미니 인스트럭트 모델
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2.27B 모델, 4비트 지원
] # 더 많은 모델은 https://huggingface.co/unsloth 에서 확인 가능

# 사전 학습된 모델과 토크나이저 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",  # 사용할 모델 이름
    max_seq_length = max_seq_length,         # 설정한 최대 시퀀스 길이
    dtype = dtype,                           # 데이터 타입 설정
    load_in_4bit = load_in_4bit,             # 4비트 양자화 여부
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.7: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [4]:
# 목표: 메모리 절약 + 속도 중시
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,   # 🔹 LoRA rank를 낮게 설정 — 작은 데이터셋엔 과적합 방지됨
    target_modules = ["q_proj", "v_proj"],  # 🔹 q/v projection만 학습 → 메모리 절감
    lora_alpha = 16,
    lora_dropout = 0.05,   # 약간의 regularization 효과
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 🔹 메모리 절약
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
alpaca_prompt = """아래에는 하나의 질문과 이에 대한 적절한 답변을 작성해야 하는 지시문이 있습니다.
지시문을 읽고, 요청에 가장 잘 맞는 답변을 한국어로 작성하세요.

### 지시문(Instruction):
{}

### 답변(Response):
{}"""

EOS_TOKEN = tokenizer.eos_token

In [16]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1️⃣ 원본 데이터 로드
dataset = load_dataset("ChuGyouk/PubMedQA-test-Ko", split="test")

# 2️⃣ 앞에서 100개만 샘플링
dataset = dataset.select(range(100))
print(f"샘플 개수: {len(dataset)}")
print(dataset[0])

샘플 개수: 100
{'QUESTION': '항문 직장 내 초음파 검사는 배변 장애에 유용합니까?', 'LONG_ANSWER': '선형 항문 직장 내시경 검사는 배변 운동 중 항문 괄약근과 항문 직장근의 이완이 불완전하거나 심지어 부재 한 것으로 나타났습니다. 이 연구는 "골반저근긴장이상증" 또는 "이완불능증"의 진단에 있어 이 우아한 초음파 촬영 기법의 가치를 강조합니다.', 'reasoning_required_pred': '예', 'reasoning_free_pred': '예', 'final_decision': '예', 'CONTEXTS': ['배변 장애는 부적절한 배변 운동으로 인해 유발 될 수 있습니다. 이 전향적 연구의 목적은 항문 직장 내 초음파 검사를 사용하여 배변 장애가있는 환자에서 항문 괄약근 및 / 또는 항문 직장 근육의 기능 장애를 입증하는 것이 었습니다.', '배뇨장애 병력이 있는 20명의 연속적인 환자와 20명의 건강한 대조군을 대상으로 선형 항문직장 내시경 검사(Toshiba 모델 IUV 5060 및 PVL-625 RT)를 시행했습니다. 두 그룹 모두에서 항문 괄약근과 치직근의 치수를 휴식 상태와 자발적으로 힘을 주거나 긴장하는 동안 측정했습니다. 통계 분석은 두 그룹 내에서 그리고 두 그룹간에 수행되었습니다.', '항문 괄약근은 85%의 환자에서 (휴식 상태와 비교하여) 긴장하는 동안 역설적으로 짧아지고/또는 두꺼워졌지만 대조군에서는 35%만 괄약근이 짧아졌습니다. 괄약근 길이의 변화는 대조군에 비해 환자군에서 통계적으로 유의미한 차이가 있었습니다(p<0.01, chi(2) 테스트). 환자의 80%는 긴장하는 동안 역설적으로 괄약근이 짧아지고/또는 두꺼워졌지만, 대조군은 30%에 불과했습니다. 환자군과 대조군 피험자에서 치핵의 길이와 두께의 변화는 모두 유의미한 차이를 보였습니다(p<0.01, chi(2) 테스트).'], 'YEAR': 2002.0}


In [17]:
# 2️⃣ 프롬프트 변환
def format_prompt(example):
    instruction = example["QUESTION"]
    response = example["LONG_ANSWER"]
    text = alpaca_prompt.format(instruction, response) + EOS_TOKEN
    return {"text": text}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [18]:
# 3️⃣ 토크나이징
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=2048,
    )

tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [9]:
# 4️⃣ 데이터 정리
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

In [12]:
!pip install -U transformers accelerate peft trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.2
    Uninstalling transformers-4.56.2:
      Successfully uninstalled transformers-4.56.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.10.1
    Uninstalling accelerate-1.10.1:
      Successfully uninstalled accelerate-1.10.1
  Attempting uninstall: trl
    Found existing installation: trl 0.23.0
    Uninstalling trl-0.23.0:
      Successfully uninstalled trl-0.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zo

In [23]:
# 5️⃣ 학습 설정
training_args = TrainingArguments(
    output_dir="./finetuned_pubmedqa",
    num_train_epochs=3,
    per_device_train_batch_size=1,   # VRAM 한계에 따라 조정
    gradient_accumulation_steps=4,   # 배치 작을 때 효과적
    warmup_steps=50,
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    fp16=True,                       # 4bit 양자화 모델은 필수
    optim="paged_adamw_8bit",        # 기울기 공간을 압축 (2~3배 절약)
    lr_scheduler_type="cosine",
    # evaluation_strategy="no",
    report_to="none"
)

In [24]:
# 6️⃣ 데이터 콜레이터 (언어모델 학습용)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [25]:
# 7️⃣ Trainer 설정
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

In [26]:
# 8️⃣ 파인튜닝 실행
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,407,872 of 8,033,669,120 (0.04% trained)


Step,Training Loss
50,2.264500


TrainOutput(global_step=75, training_loss=2.0406744384765627, metrics={'train_runtime': 1395.9735, 'train_samples_per_second': 0.215, 'train_steps_per_second': 0.054, 'total_flos': 2.76787170902016e+16, 'train_loss': 2.0406744384765627, 'epoch': 3.0})

In [27]:
# 9️⃣ 학습 완료 후 저장
model.save_pretrained("./finetuned_pubmedqa_model")
tokenizer.save_pretrained("./finetuned_pubmedqa_model")

('./finetuned_pubmedqa_model/tokenizer_config.json',
 './finetuned_pubmedqa_model/special_tokens_map.json',
 './finetuned_pubmedqa_model/tokenizer.json')